# Exercise 4 — validate_api_key and gate_feature

Feature gating is how you enforce paid access to your product. `validate_api_key` checks whether a key is in the set of valid keys. `gate_feature` wraps any function with an access check — valid key allows the call, invalid key raises `PermissionError`. This is the minimal access control layer before adding a real billing system.

In [ ]:
from dataclasses import dataclass, field
import datetime

@dataclass
class ProductConfig:
    name: str; version: str; description: str
    author: str; email: str
    dependencies: list = field(default_factory=list)
    license: str  = "MIT"
    python_requires: str = ">=3.10"

_CFG = ProductConfig(
    name="my-trading-bot",
    version="0.1.0",
    description="AI-powered paper-trading bot using sentiment and technical signals.",
    author="Jane Doe",
    email="jane@example.com",
    dependencies=["pandas>=2.0", "requests>=2.28"],
)

def validate_api_key(key, valid_keys):
    """True if key is in valid_keys (case-sensitive exact match)."""
    # TODO: 1 line
    return False


def gate_feature(api_key, valid_keys, feature_fn, *args, **kwargs):
    """Call feature_fn(*args, **kwargs) only if api_key is valid.

    Raises PermissionError if the key is not in valid_keys.
    """
    # TODO: ~3 lines
    return None


### Checks

In [ ]:
checks = 0
KEYS = {"key-abc-123", "key-xyz-456"}

# 1 — valid key returns True
try:
    assert validate_api_key("key-abc-123", KEYS) is True
    checks += 1; print("✅ 1 valid key → True")
except Exception as e:
    print("❌ 1:", e)

# 2 — invalid key returns False
try:
    assert validate_api_key("bad-key", KEYS) is False
    assert validate_api_key("KEY-ABC-123", KEYS) is False  # case-sensitive
    checks += 1; print("✅ 2 invalid/wrong-case key → False")
except Exception as e:
    print("❌ 2:", e)

# 3 — gate_feature calls fn with valid key
try:
    result = gate_feature("key-abc-123", KEYS, lambda x: x * 2, 21)
    assert result == 42, f"expected 42, got {result}"
    checks += 1; print("✅ 3 gate_feature calls fn(21) → 42 with valid key")
except Exception as e:
    print("❌ 3:", e)

# 4 — gate_feature raises PermissionError for invalid key
try:
    try:
        gate_feature("bad-key", KEYS, lambda: "secret")
        print("❌ 4: expected PermissionError")
    except PermissionError as pe:
        assert "bad-key" in str(pe), f"PermissionError should mention the bad key: {pe}"
        checks += 1; print("✅ 4 invalid key → PermissionError (mentions the key)")
except Exception as e:
    print("❌ 4:", e)

# 5 — gate_feature passes kwargs to fn
try:
    def multiply(x, factor=1):
        return x * factor
    result = gate_feature("key-xyz-456", KEYS, multiply, 5, factor=10)
    assert result == 50, f"expected 50, got {result}"
    checks += 1; print("✅ 5 gate_feature passes *args and **kwargs to fn")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
